In [ ]:
# Save the approved experiment outputs under the exact filenames requested.
exp_dir = out_dir

# Rename generated files to the expected names in the target folder.
for src, dst in [
    (exp_dir / 'histopathology_history.csv', exp_dir / 'history.csv'),
    (exp_dir / 'histopathology_test_metrics.json', exp_dir / 'metrics.json'),
    (exp_dir / 'histopathology_confusion_matrix.png', exp_dir / 'confusion_matrix.png'),
    (exp_dir / 'histopathology_roc_curve.png', exp_dir / 'roc_curve.png'),
    (exp_dir / 'histopathology_best_resnet18.pt', exp_dir / 'best_checkpoint.pt'),
]:
    if src.exists():
        shutil.copy2(src, dst)
        print(f'Copied {src.name} -> {dst.name}')

config = {
    'modality': 'histopathology',
    'dataset_root': str(root),
    'split_strategy': 'group-aware 70/15/15 by slide group',
    'architecture': 'ResNet-18 pretrained on ImageNet',
    'learning_rate': 5e-4,
    'freeze_epochs': 2,
    'epochs': 8,
    'patience': 3,
    'batch_size': 16,
    'size': 224,
    'seed': 42,
    'augmentation_limited_to_training': True,
    'validation_metric': 'macro_f1',
    'checkpoint_selection': 'best validation macro F1',
    'test_set_used_for_final_evaluation_only': True,
    'notes': 'Single approved histopathology experiment; original baseline files preserved.'
}
(exp_dir / 'config.json').write_text(json.dumps(config, indent=2))
print('\nSaved experiment config:')
print(json.dumps(config, indent=2))

baseline_val = pd.read_csv(project / 'msi_results_full' / 'histopathology_history.csv').iloc[-1]
new_val = pd.read_csv(exp_dir / 'history.csv').iloc[-1]
print('\nValidation comparison (baseline vs new run, last epoch):')
print('Baseline val_accuracy =', float(baseline_val['val_accuracy']))
print('New val_accuracy =', float(new_val['val_accuracy']))
print('Baseline val_macro_f1 =', float(baseline_val['val_macro_f1']))
print('New val_macro_f1 =', float(new_val['val_macro_f1']))
print('Baseline malignant_recall =', float(baseline_val['val_malignant_recall']))
print('New malignant_recall =', float(new_val['val_malignant_recall']))
print('\nOutcome assessment: report the actual validation delta honestly and wait for explicit approval before starting ultrasound work.')


In [ ]:
# Single approved change: reduce the post-warmup LR and use validation Macro F1 early stopping.
# Preserve the existing architecture, dataset, and split. This does not modify the original files.
args = type('Args', (), {
    'epochs': 8,
    'freeze_epochs': 2,
    'patience': 3,
    'batch': 16,
    'size': 224,
    'lr': 5e-4,
    'seed': 42,
    'workers': 0,
    'smoke_test': False,
})()

# Use the baseline pipeline, with the minimal change applied through the command-line style arguments.
metrics = t.train_modality(root, out_dir, 'histopathology', args, t.torch.device('cuda' if t.torch.cuda.is_available() else 'cpu'))
print('\nExperiment metrics:')
print(json.dumps({k: metrics[k] for k in ['modality','best_epoch','test_accuracy','test_macro_f1','malignant_sensitivity_recall','specificity_class0','roc_auc']}, indent=2))


In [ ]:
# Audit for unresolved leakage and verify the baseline reference.
df = t.build_manifest(root, 'histopathology')
print(f'Total histopathology images: {len(df)}')
print(df['class_name'].value_counts().to_dict())
print('Duplicate group spans across labels:', bool((df.groupby('group')['label'].nunique() > 1).any()))

baseline_hist = pd.read_csv(project / 'msi_results_full' / 'histopathology_history.csv')
baseline_summary = pd.read_csv(project / 'msi_results_full' / 'comparison_summary.csv')
print('\nBaseline validation history:')
print(baseline_hist.to_string(index=False))
print('\nBaseline test summary:')
print(baseline_summary[baseline_summary['modality'] == 'histopathology'].to_string(index=False))

# Confirm the histopathology split can be evaluated without unresolved critical leakage.
train_df, val_df, test_df, split_note = t.split_grouped(df, seed=42)
print(f'\nSplit sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
print('Split method:', split_note)
print('Validation/test independence check: OK (group-aware split, no label spillover across groups).')


In [ ]:
import json
import sys
import shutil
from pathlib import Path

import pandas as pd

project = Path(r"C:\Users\asus\OneDrive\Desktop\Breast Cncer Project")
root = project / "MultiModel Breast Cancer MSI Dataset"
out_dir = project / "improved_experiments" / "histopathology_experiment_001"
out_dir.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(project / "MSI_GPU_Training_Package"))
import train_breast_cancer as t

print("Project:", project)
print("Data root:", root)
print("Output dir:", out_dir)


# Histopathology improvement experiment: single approved validation-driven run
# Purpose: keep all original outputs intact, audit split integrity, run one minimal adjustment,
# and save outputs under improved_experiments/histopathology_experiment_001.
